# Description

Reads the *full predictions* (all traits, not just DOID-mapped) from the 245 module-based HDF5 files
generated by `011-prediction-gene_module_based.ipynb` and saves them per-tissue for later use.

Mirrors `phenoplier/nbs/30_drug_disease_associations/100-lincs/039-save_full_predictions.ipynb` exactly.

For each of 49 tissues:
1. Loads all 5 threshold files (all_genes, top_5, top_10, top_25, top_50).
2. Ranks scores within each file.
3. Sums ranks across the 5 thresholds.
4. Divides by 5 to get the average rank.
5. Saves the resulting (4091 traits × 1170 drugs) float32 DataFrame to HDF5, keyed by tissue name.

**Input**: `output/drug_disease_analyses/lincs/predictions/dotprod_neg/` (245 module-based `.h5` files)  
**Output**: `output/drug_disease_analyses/lincs/predictions/full_predictions_by_tissue-rank.h5`

# Modules loading

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd
from tqdm import tqdm

from pyprojroot import here

# Settings

In [3]:
# These numbers are for checking
N_TISSUES = 49
N_THRESHOLDS = 5

In [4]:
INPUT_DIR = here('output/drug_disease_analyses') / 'lincs'
display(INPUT_DIR)
assert INPUT_DIR.exists()

INPUT_PREDICTIONS_DIR = INPUT_DIR / 'predictions' / 'dotprod_neg'
display(INPUT_PREDICTIONS_DIR)
assert INPUT_PREDICTIONS_DIR.exists()

OUTPUT_DIR = INPUT_DIR / 'predictions'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
display(OUTPUT_DIR)

OUTPUT_FILENAME = OUTPUT_DIR / 'full_predictions_by_tissue-rank.h5'
display(OUTPUT_FILENAME)

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/lincs')

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/lincs/predictions/dotprod_neg')

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/lincs/predictions')

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/lincs/predictions/full_predictions_by_tissue-rank.h5')

# Load drug-disease predictions

In [5]:
# Module-based files have '-projection-' in their name
current_prediction_files = sorted(
    f for f in INPUT_PREDICTIONS_DIR.glob('*.h5') if '-projection-' in f.name
)
display(len(current_prediction_files))

assert len(current_prediction_files) == N_TISSUES * N_THRESHOLDS, (
    f'Expected {N_TISSUES * N_THRESHOLDS} files, found {len(current_prediction_files)}.\n'
    'Run 011-prediction-gene_module_based.ipynb first.'
)

245

In [6]:
current_prediction_files[:5]

[PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/lincs/predictions/dotprod_neg/spredixcan-mashr-zscores-Adipose_Subcutaneous-projection-all_genes-prediction_scores.h5'),
 PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/lincs/predictions/dotprod_neg/spredixcan-mashr-zscores-Adipose_Subcutaneous-projection-top_10_genes-prediction_scores.h5'),
 PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/lincs/predictions/dotprod_neg/spredixcan-mashr-zscores-Adipose_Subcutaneous-projection-top_25_genes-prediction_scores.h5'),
 PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/lincs/predictions/dotprod_neg/spredixcan-mashr-zscores-Adipose_Subcutaneous-projection-top_50_genes-prediction_scores.h5'),
 PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/lincs/predictions/dotprod_neg/spredixcan-mashr-zscores-Adipose

In [7]:
def _get_tissue(data_value):
    """
    Extracts tissue name from the metadata 'data' field.
    E.g.: 'spredixcan-mashr-zscores-Liver-projection' -> 'Liver'
    """
    if data_value.endswith('-projection'):
        return data_value.split('spredixcan-mashr-zscores-')[1].split('-projection')[0]
    else:
        return data_value.split('spredixcan-mashr-zscores-')[1].split('-data')[0]

In [8]:
# Collect all tissue names and threshold values
all_tissues = set()
all_methods = set()

for f in tqdm(current_prediction_files, ncols=100):
    metadata = pd.read_hdf(f, key='metadata')
    _data = metadata['data'].values[0]
    all_tissues.add(_get_tissue(_data))
    all_methods.add(metadata['n_top_genes'].values[0])

assert len(all_methods) == N_THRESHOLDS, f'Expected {N_THRESHOLDS} thresholds, got {len(all_methods)}'
display(f'Thresholds: {sorted(all_methods)}')

  0%|                                                                       | 0/245 [00:00<?, ?it/s]

  6%|███▍                                                         | 14/245 [00:00<00:01, 138.28it/s]

 16%|█████████▉                                                   | 40/245 [00:00<00:00, 205.51it/s]

 27%|████████████████▍                                            | 66/245 [00:00<00:00, 226.88it/s]

 38%|██████████████████████▉                                      | 92/245 [00:00<00:00, 236.56it/s]

...

 69%|█████████████████████████████████████████▍                  | 169/245 [00:00<00:00, 246.43it/s]

 80%|███████████████████████████████████████████████▊            | 195/245 [00:00<00:00, 247.62it/s]

 90%|██████████████████████████████████████████████████████      | 221/245 [00:00<00:00, 248.91it/s]

100%|████████████████████████████████████████████████████████████| 245/245 [00:01<00:00, 239.09it/s]

'Thresholds: [np.float64(-1.0), np.float64(5.0), np.float64(10.0), np.float64(25.0), np.float64(50.0)]'

In [9]:
all_tissues = sorted(all_tissues)
assert len(all_tissues) == N_TISSUES
display(f'Tissues found: {len(all_tissues)}')

'Tissues found: 49'

In [10]:
# Get all trait and drug names from the first file
_tmp_df = pd.read_hdf(current_prediction_files[0], key='full_prediction')
all_traits = _tmp_df['trait'].drop_duplicates().tolist()
all_drugs = _tmp_df['drug'].drop_duplicates().tolist()
display(f'Traits: {len(all_traits)}, Drugs: {len(all_drugs)}')

'Traits: 4091, Drugs: 1170'

In [11]:
assert len(all_traits) == 4091
assert len(all_drugs) == 1170

# Create per-tissue averaged rank predictions

For each tissue:
1. Load the 5 threshold files.
2. Rank scores within each file.
3. Sum and average ranks across the 5 thresholds.
4. Save the result (4091 × 1170) to HDF5 keyed by tissue name.

In [12]:
with pd.HDFStore(OUTPUT_FILENAME, mode='w', complevel=4) as store:
    for tissue in tqdm(all_tissues, ncols=100):
        # Get all prediction files for this tissue
        tissue_prediction_files = [
            f for f in current_prediction_files if f'-{tissue}-' in f.name
        ]
        assert len(tissue_prediction_files) == len(all_methods), (
            f'Tissue {tissue}: expected {len(all_methods)} files, found {len(tissue_prediction_files)}'
        )

        # Initialize accumulator
        tissue_df = pd.DataFrame(
            data=0,
            index=all_traits.copy(),
            columns=all_drugs.copy(),
            dtype='float32',
        )

        for f in tissue_prediction_files:
            # Verify tissue matches
            metadata = pd.read_hdf(f, key='metadata')
            _data = metadata['data'].values[0]
            assert _get_tissue(_data) == tissue

            # Load full prediction, rank scores, pivot to traits × drugs
            prediction_data = pd.read_hdf(f, key='full_prediction')
            prediction_data['score'] = prediction_data['score'].rank()
            prediction_data = prediction_data.pivot(
                index='trait', columns='drug', values='score'
            ).astype('float32')

            # Accumulate (sum across N_THRESHOLDS)
            tissue_df += prediction_data.loc[tissue_df.index, tissue_df.columns]

        # Save the average rank (divide by number of thresholds)
        store.put(
            tissue,
            (tissue_df / len(all_methods)).astype('float32'),
            format='fixed',
        )

print(f'Saved to: {OUTPUT_FILENAME}')

  0%|                                                                        | 0/49 [00:00<?, ?it/s]

  2%|█▎                                                              | 1/49 [00:07<06:08,  7.69s/it]

  4%|██▌                                                             | 2/49 [00:15<05:57,  7.61s/it]

  6%|███▉                                                            | 3/49 [00:23<05:59,  7.82s/it]

  8%|█████▏                                                          | 4/49 [00:31<05:52,  7.84s/it]

...

 96%|████████████████████████████████████████████████████████████▍  | 47/49 [06:11<00:15,  7.72s/it]

 98%|█████████████████████████████████████████████████████████████▋ | 48/49 [06:18<00:07,  7.62s/it]

100%|███████████████████████████████████████████████████████████████| 49/49 [06:26<00:00,  7.56s/it]

100%|███████████████████████████████████████████████████████████████| 49/49 [06:26<00:00,  7.88s/it]

Saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/lincs/predictions/full_predictions_by_tissue-rank.h5


# Testing

In [13]:
_tissue = 'Adipose_Subcutaneous'

with pd.HDFStore(OUTPUT_FILENAME, mode='r') as store:
    tissue_df = store[_tissue]

display(tissue_df.shape)
assert tissue_df.shape == (4091, 1170)
assert not tissue_df.isna().any().any()
display(tissue_df.head())

(4091, 1170)

,DB00014,DB00091,DB00121,DB00130,DB00131,DB00132,DB00136,DB00140,DB00146,DB00150,...,DB08995,DB09002,DB09004,DB09009,DB09010,DB09015,DB09019,DB09020,DB09022,DB09023
100001_raw_Food_weight,2326938.25,2.605673e+06,1492423.125,1495564.25,2401406.50,1781544.000,2369688.000,2670029.50,3438604.750,3513419.25,...,1116372.00,2222350.50,2779125.50,2079129.75,1298991.00,2191118.000,3467686.500,2626670.500,3077448.25,2949800.25
100002_raw_Energy,3666857.25,2.613616e+05,2680665.500,2057881.25,2484043.00,3107461.000,2008964.375,2489074.25,1744926.625,2331236.50,...,3399969.50,3999509.25,3385979.25,2114243.75,1921787.75,2092250.625,2394991.000,1991220.000,3070348.75,3617594.50
100003_raw_Protein,2170337.25,9.681392e+05,1148000.875,3043983.50,3222495.75,1476908.375,3014119.500,1843406.25,910908.375,1711716.75,...,2811056.50,3874197.50,3819008.00,3139775.00,1737145.75,3581618.000,3054337.000,4328281.500,2527716.50,3341164.50
100004_raw_Fat,3377385.50,3.269700e+04,1441765.500,2295014.50,2617964.00,3205508.750,300374.500,3211276.50,385624.000,2830960.00,...,2870610.50,4053461.25,3642456.75,2020456.75,1938754.00,2451948.000,3175998.750,4697709.000,1958132.00,2162544.75
100005_raw_Carbohydrate,3078809.50,3.579820e+05,2597569.500,2933452.50,2628369.75,2408882.500,2206940.500,3160994.75,2572691.500,1832625.00,...,2252888.75,3449544.00,2582148.00,2642699.75,2317894.75,2459056.500,1963910.375,1158200.375,3531360.50,3832612.75


In [14]:
# Verify: manually check one tissue/drug/trait combination
_files = [f for f in current_prediction_files if f'-{_tissue}-' in f.name]
display(len(_files))
assert len(_files) == N_THRESHOLDS

_files_data = [
    pd.read_hdf(f, key='full_prediction').set_index(['trait', 'drug'])['score'].rank()
    for f in _files
]

_trait = all_traits[0]
_drug = all_drugs[0]

_expected = np.mean([s.loc[(_trait, _drug)] for s in _files_data])
_actual = tissue_df.loc[_trait, _drug]

display(f'Expected: {_expected:.4f}, Actual: {float(_actual):.4f}')
assert abs(_expected - float(_actual)) < 1.0  # float32 rounding tolerance

5

'Expected: 2326938.2000, Actual: 2326938.2500'

In [15]:
# List all keys in the output HDF5
with pd.HDFStore(OUTPUT_FILENAME, mode='r') as store:
    keys = store.keys()

display(f'Total tissues saved: {len(keys)}')
assert len(keys) == N_TISSUES

'Total tissues saved: 49'